In [19]:
import os
import pathlib
import json
import pandas as pd
from openai import OpenAI

client = OpenAI()

In [5]:
# gpt-5-nano 모델 호출 예시
def ask_direct(question: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"{question} 최종 답만 숫자로 출력하라."}],
        # temperature=0,
    )
    return resp.choices[0].message.content

print(ask_direct("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))
# → "213000"   ← 틀림! (정답은 213300)

213300


In [13]:
# CoT 적용 예시
def ask_cot(question: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"{question} 생각 과정을 단계별로 작성한 뒤 마지막에 정답을 출력하라."}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(ask_cot("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))

1. **이어버드 가격 확인**: 이어버드 하나의 가격은 79,000원입니다.

2. **구매 개수 확인**: 이어버드를 3개 구매했습니다.

3. **총 가격 계산**: 
   - 총 가격 = 이어버드 가격 × 구매 개수
   - 총 가격 = 79,000원 × 3 = 237,000원

4. **쿠폰 할인율 확인**: 10% 쿠폰을 받았습니다.

5. **할인 금액 계산**: 
   - 할인 금액 = 총 가격 × 할인율
   - 할인 금액 = 237,000원 × 0.10 = 23,700원

6. **최종 결제액 계산**: 
   - 최종 결제액 = 총 가격 - 할인 금액
   - 최종 결제액 = 237,000원 - 23,700원 = 213,300원

따라서, 총 결제액은 **213,300원**입니다.


In [14]:
def ask_cot(question: str) -> str:
    """CoT: 한 줄씩 풀이를 쓰게 한다.

    [왜] 모델은 앞서 쓴 자기 출력을 다시 입력으로 참고한다. 풀이를 글로 쓰게 하면
    그 풀이가 다음 토큰 생성의 '작업 공간(근거)'이 되어 마지막 답이 정확해진다.
    """
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"{question}\n"
                    "단계적으로 풀어라. 각 계산을 한 줄씩 쓰고, "
                    "맨 마지막 줄에 '정답: <숫자>' 형식으로 답하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [15]:
import re

def extract_number(text: str):
    """답변에서 마지막 숫자를 정수로 반환(콤마 제거)."""
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    if not nums:
        return None
    return int(nums[-1].replace(",", ""))

In [20]:
q = "이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"
print("[직접] ", ask_direct(q))
print("[CoT]\n", ask_cot(q))

[직접]  213300
[CoT]
 1. 이어버드 한 개의 가격: 79,000원  
2. 이어버드 3개의 가격: 79,000원 × 3 = 237,000원  
3. 10% 쿠폰 할인액: 237,000원 × 0.10 = 23,700원  
4. 총 결제액: 237,000원 - 23,700원 = 213,300원  

정답: 213300


In [21]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def ask_direct(question: str) -> str:
    """직접 답변: 중간 과정 없이 최종 숫자만 시킨다 → 다단계 계산에서 자주 틀림."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 최종 숫자(원)만 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [22]:
q = "이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"
first = ask_cot(q)          # 1차 풀이
print("[검산 결과]\n", verify(q, first))   # 2차 검산

NameError: name 'verify' is not defined

In [23]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-nano"

def verify(question: str, cot_answer: str) -> str:
    """제출한 풀이를 모델에게 다시 검산시켜 신뢰도를 높인다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"[문제] {question}\n[제출한 풀이]\n{cot_answer}\n\n"
                    "위 풀이가 맞는지 다시 계산해 검산하라. "
                    "틀렸다면 올바른 값을, 맞다면 그대로 '정답: <숫자>'로 확정하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [24]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def verify(question: str, cot_answer: str) -> str:
    """제출한 풀이를 모델에게 다시 검산시켜 신뢰도를 높인다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"[문제] {question}\n[제출한 풀이]\n{cot_answer}\n\n"
                    "위 풀이가 맞는지 다시 계산해 검산하라. "
                    "틀렸다면 올바른 값을, 맞다면 그대로 '정답: <숫자>'로 확정하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [25]:
wrong_solution = """1) 79000 × 3 = 237000
2) 237000 × 10% = 23700
3) 237000 - 23700 = 213000   ← 계산 실수!
정답: 213000"""

print(verify(q, wrong_solution))

주어진 문제를 다시 계산해 보겠습니다.

1) 이어버드의 가격은 79,000원이므로, 3개를 구매하면:
   \[
   79,000 \times 3 = 237,000 \text{원}
   \]

2) 10% 쿠폰을 적용하면 할인액은:
   \[
   237,000 \times 0.10 = 23,700 \text{원}
   \]

3) 최종 결제액은:
   \[
   237,000 - 23,700 = 213,300 \text{원}
   \]

따라서, 제출한 풀이에서 계산이 잘못되었습니다. 올바른 결제액은 213,300원입니다.

정답: 213300


In [26]:
def answer_with_optional_verify(question: str, critical: bool) -> str:
    """critical=True(돈·재고 등 민감)일 때만 검산을 추가한다."""
    first = ask_cot(question)
    if critical:
        return verify(question, first)   # 민감한 계산 → 검산
    return first                          # 일반 질문 → 그대로

In [27]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-5-nano"

def ask_direct(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 정답만 한 단어로 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens

def ask_cot(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n단계적으로 풀어라. 마지막 줄에 '정답: <값>'."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens

d_txt, d_tok = ask_direct("대한민국의 수도는 어디인가?")
c_txt, c_tok = ask_cot("대한민국의 수도는 어디인가?")
print(f"직접: {d_txt} (토큰 {d_tok})")
print(f"CoT : 토큰 {c_tok}  ← 같은 정답인데 토큰만 더 씀")

BadRequestError: Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0 with this model. Only the default (1) value is supported.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'unsupported_value'}}

In [ ]:
cases = [
    ("배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?", "50"),
    ("5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?", "5"),  # 정답 5분
]

for q, gold in cases:
    d_txt, _ = ask_direct(q)
    c_txt, _ = ask_cot(q)
    print(f"Q: {q}")
    print(f"  정답: {gold} / 직접: {d_txt[:20]} / CoT 마지막줄: {c_txt.splitlines()[-1]}")